# 03 — Analysis: So sánh 4 frameworks và vẽ biểu đồ báo cáo

Notebook này đọc kết quả từ `results/metrics/` và tạo các biểu đồ dùng trong báo cáo IEEE.

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

# FIX #9: dùng path tuyệt đối từ vị trí notebook, tránh lỗi khi CWD không phải notebooks/
NOTEBOOK_DIR = Path(os.path.abspath(''))
ROOT_DIR     = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
METRICS_DIR  = ROOT_DIR / 'results' / 'metrics'
FIGURES_DIR  = ROOT_DIR / 'results' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'ROOT_DIR   : {ROOT_DIR}')
print(f'METRICS_DIR: {METRICS_DIR}')
print(f'FIGURES_DIR: {FIGURES_DIR}')

# FIX #3/#4: chuẩn hoá tên framework — tất cả dùng titlecase nhất quán
# Key phải khớp với giá trị 'framework' trong các file JSON
FRAMEWORK_COLORS = {
    'Ray':     '#4C72B0',
    'Spark':   '#DD8452',
    'MindsDB': '#55A868',
    'vLLM':    '#C44E52',
}

# Map để chuẩn hoá tên bất kể case/format trong JSON
FRAMEWORK_NORMALIZE = {
    'ray':     'Ray',
    'spark':   'Spark',
    'mindsdb': 'MindsDB',
    'vllm':    'vLLM',
}

def normalize_fw(name: str) -> str:
    """Chuẩn hoá tên framework về dạng chuẩn (Ray/Spark/MindsDB/vLLM)."""
    return FRAMEWORK_NORMALIZE.get(str(name).lower().strip(), name)

In [ ]:
# Load tất cả benchmark files
def load_all_benchmarks(metrics_dir: Path) -> pd.DataFrame:
    rows = []
    for fp in sorted(metrics_dir.glob('benchmark_*.json')):
        with open(fp) as f:
            d = json.load(f)
        # FIX #3: chuẩn hoá tên framework ngay khi load
        d['framework'] = normalize_fw(d.get('framework', fp.stem))
        rows.append(d)
    return pd.DataFrame(rows) if rows else pd.DataFrame()

# FIX #5: Ray lưu ra eval_metrics.json, không phải eval_ray.json
# Map tên framework -> tên file eval thực tế
EVAL_FILE_MAP = {
    'ray':     'eval_metrics.json',   # src/evaluate.py lưu ra đây
    'spark':   'eval_spark.json',
    'mindsdb': 'eval_mindsdb.json',
    'vllm':    'eval_vllm.json',
}

def load_eval(framework: str) -> dict:
    """Load eval metrics cho một framework, tự tìm đúng tên file."""
    key = framework.lower().strip()
    filename = EVAL_FILE_MAP.get(key, f'eval_{key}.json')
    fp = METRICS_DIR / filename
    if fp.exists():
        with open(fp) as f:
            d = json.load(f)
        d['framework'] = normalize_fw(d.get('framework', framework))
        return d
    return {}

bench_df = load_all_benchmarks(METRICS_DIR)

# FIX #1: guard khi chưa có kết quả
if bench_df.empty:
    print('⚠️  Chưa có benchmark files trong', METRICS_DIR)
    print('   Chạy experiments/ trước: python experiments/run_ray.py, run_spark.py, ...')
else:
    # FIX #2: dùng reindex để tránh KeyError khi cột thiếu
    display_cols = ['framework', 'throughput_mean', 'throughput_std', 'total_train_time']
    print(bench_df.reindex(columns=display_cols).to_string(index=False))

## Figure 1: ETL Throughput so sánh

In [ ]:
if bench_df.empty:
    print('Bỏ qua Figure 1 — chưa có benchmark data.')
else:
    fig, ax = plt.subplots(figsize=(8, 4))

    frameworks  = bench_df['framework'].tolist()
    throughputs = bench_df['throughput_mean'].fillna(0).tolist()
    colors      = [FRAMEWORK_COLORS.get(f, '#888888') for f in frameworks]

    # FIX #2: throughput_std có thể không tồn tại trong một số JSON
    if 'throughput_std' in bench_df.columns:
        errors = bench_df['throughput_std'].fillna(0).tolist()
    else:
        errors = [0] * len(frameworks)

    bars = ax.bar(frameworks, throughputs, color=colors, alpha=0.85,
                  yerr=errors, capsize=4, edgecolor='white', linewidth=0.5)

    ax.set_ylabel('Throughput (samples/sec)')
    ax.set_title('ETL Throughput Comparison — AG News (120k samples)')

    for bar, val in zip(bars, throughputs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                f'{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig1_etl_throughput.png', bbox_inches='tight')
    plt.show()
    print(f'Saved → {FIGURES_DIR}/fig1_etl_throughput.png')

## Figure 2: Accuracy vs Training Time

In [ ]:
# Load eval metrics cho cả 4 framework
eval_data = []
for fw in ['ray', 'spark', 'mindsdb', 'vllm']:
    d = load_eval(fw)
    if d:
        eval_data.append(d)
    else:
        print(f'⚠️  Không tìm thấy eval file cho {fw} ({EVAL_FILE_MAP.get(fw)})')

eval_df = pd.DataFrame(eval_data)

if eval_df.empty or bench_df.empty:
    print('Bỏ qua Figure 2 — cần cả eval metrics và benchmark data.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))

    for _, row in eval_df.iterrows():
        fw    = row.get('framework', '?')   # đã được normalize khi load
        acc   = row.get('accuracy', 0)
        color = FRAMEWORK_COLORS.get(fw, '#888888')

        # FIX #3: fw đã được normalize (Ray/Spark/...) → match đúng với bench_df
        t_vals = bench_df[bench_df['framework'] == fw]['total_train_time'].values
        t = float(t_vals[0]) if len(t_vals) > 0 and not np.isnan(t_vals[0]) else 0.0

        ax.scatter(t, acc, s=180, color=color, zorder=5,
                   label=fw, edgecolors='white', linewidths=1.5)
        ax.annotate(fw, (t, acc), textcoords='offset points',
                    xytext=(8, 4), fontsize=10)

    ax.set_xlabel('Training time (seconds)')
    ax.set_ylabel('Test Accuracy')
    ax.set_title('Accuracy vs Training Time')
    ax.legend(loc='lower right')

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig2_accuracy_vs_time.png', bbox_inches='tight')
    plt.show()
    print(f'Saved → {FIGURES_DIR}/fig2_accuracy_vs_time.png')

## Figure 3: Radar chart — 5 tiêu chí đánh giá

In [ ]:
# FIX #6: thêm comment nguồn tham khảo cho từng điểm
# Thang điểm 1-5, dựa trên:
#   Performance/Scalability : benchmark thực nghiệm (throughput, train time)
#   Ease of Use             : tài liệu chính thức + độ phức tạp setup
#   Flexibility             : số loại task/data type hỗ trợ
#   Real-World Adoption     : số GitHub stars, paper citations, industry surveys
#   AI/Automation           : native LLM/AutoML support
#
# Nguồn:
#   Ray        : Moritz et al. 2018; docs.ray.io; benchmarks thực nghiệm
#   Spark      : Zaharia et al. 2016; Databricks State of Data+AI 2023
#   MindsDB    : MindsDB docs 2024; "AI-Native DB" whitepaper
#   vLLM       : Kwon et al. 2023; vLLM GitHub benchmarks

CRITERIA = [
    'Performance &\nScalability',
    'Ease of Use &\nSetup',
    'Flexibility &\nExtensibility',
    'Real-World\nAdoption',
    'AI & Automation\nCapabilities',
]

#              [Perf, Ease, Flex, Adopt, AI]
SCORES = {
    'Ray':     [5,    3,    5,    4,     4],  # throughput đứng đầu; setup phức tạp hơn Spark
    'Spark':   [4,    3,    4,    5,     2],  # adoption cao nhất (enterprise); AI yếu nhất
    'MindsDB': [3,    5,    3,    3,     5],  # SQL interface dễ nhất; AI/AutoML tốt nhất
    'vLLM':    [4,    3,    3,    4,     5],  # inference throughput cao; chỉ dùng cho LLM
}

N      = len(CRITERIA)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # đóng polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for fw, scores in SCORES.items():
    vals  = scores + scores[:1]
    color = FRAMEWORK_COLORS[fw]
    ax.plot(angles, vals, 'o-', linewidth=2, label=fw, color=color)
    ax.fill(angles, vals, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(CRITERIA, size=9)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1', '2', '3', '4', '5'], size=8)
ax.set_ylim(0, 5)
ax.set_title('Framework Comparison — 5 Criteria (1–5 scale)', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_radar_comparison.png', bbox_inches='tight')
plt.show()
print(f'Saved → {FIGURES_DIR}/fig3_radar_comparison.png')

## Table: Bảng Evaluation cho báo cáo IEEE

In [ ]:
table_data = []
for fw, scores in SCORES.items():
    row = {'Framework': fw}
    for crit, score in zip(['Performance', 'Ease of Use', 'Flexibility', 'Adoption', 'AI/Automation'], scores):
        row[crit] = score

    # FIX #5: load_eval dùng EVAL_FILE_MAP, Ray đọc đúng eval_metrics.json
    d = load_eval(fw)  # fw đã là titlecase (Ray/Spark/...) → normalize_fw() xử lý đúng
    acc   = d.get('accuracy')
    f1    = d.get('f1_macro')
    row['Accuracy'] = f'{acc:.4f}' if isinstance(acc, float) else 'N/A'
    row['F1 Macro'] = f'{f1:.4f}'  if isinstance(f1,  float) else 'N/A'

    table_data.append(row)

table_df = pd.DataFrame(table_data).set_index('Framework')
print('=== Evaluation Table ===')
print(table_df.to_markdown())

# FIX #7: CSV là data → lưu vào METRICS_DIR, không phải FIGURES_DIR
csv_path = METRICS_DIR / 'evaluation_table.csv'
table_df.to_csv(csv_path)
print(f'\nSaved → {csv_path}')